In [ ]:
from rdflib import Graph

g = Graph()
g.parse("microclimate-sensors-data-enriched-updated.ttl", format="ttl")

q = """
SELECT ?sensor
WHERE {
  ?sensor a us:MicroclimateSensor .
}
"""

results = g.query(q)
for row in results:
    print(row)

In [7]:
from rdflib import Graph

g = Graph()
g.parse("pedestrian_counts_triplets.ttl", format="ttl")

<Graph identifier=Na56d14ba7127404a9dad71f913fac5a7 (<class 'rdflib.graph.Graph'>)>

In [ ]:
q = """
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX us: <https://smartcity.linkeddata.es/lcc/ontology/urban-sensors#>

SELECT ?observation
WHERE {
  ?observation a us:PedestrianObservation .
}
LIMIT 3

"""

results = g.query(q)
for row in results:
    print(row)

(rdflib.term.URIRef('https://smartcity.linkeddata.es/lcc/resource/Observation/872220241208'),)
(rdflib.term.URIRef('https://smartcity.linkeddata.es/lcc/resource/Observation/20320241223'),)
(rdflib.term.URIRef('https://smartcity.linkeddata.es/lcc/resource/Observation/1241520241227'),)


In [4]:
q = """
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX us: <https://smartcity.linkeddata.es/lcc/ontology/urban-sensors#>

SELECT ?observation ?sensor ?totalCount
WHERE {
  ?observation a us:PedestrianObservation ;
               sosa:madeBySensor ?sensor ;
               us:pedestrianCount ?totalCount .
}
ORDER BY DESC(?totalCount)
LIMIT 3

"""

results = g.query(q)
for row in results:
    print(row)



(rdflib.term.URIRef('https://smartcity.linkeddata.es/lcc/resource/Observation/352020241231'), rdflib.term.Literal('SouthB_T'), rdflib.term.Literal('7076', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.URIRef('https://smartcity.linkeddata.es/lcc/resource/Observation/122320241231'), rdflib.term.Literal('NewQ_T'), rdflib.term.Literal('6598', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))
(rdflib.term.URIRef('https://smartcity.linkeddata.es/lcc/resource/Observation/842320241231'), rdflib.term.Literal('ElFi_T'), rdflib.term.Literal('6528', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')))


In [10]:
q = """
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX us: <https://smartcity.linkeddata.es/lcc/ontology/urban-sensors#>

SELECT ?sensor ?count ?dir1 ?dir2 ?time
WHERE {
  ?observation a us:PedestrianObservation ;
               sosa:resultTime ?time ;
               sosa:madeBySensor ?sensor ;
               us:pedestrianCount ?dir1 ;
               us:pedestrianCountD1 ?dir2 ;
               us:pedestrianCountD2 ?count .

  FILTER (STRSTARTS(STR(?time), "2024-12-02") && ?sensor = "FliS_T")
}
ORDER BY DESC(?time)

"""

results = g.query(q)
for row in results:
    print(row)



(rdflib.term.Literal('FliS_T'), rdflib.term.Literal('306', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')), rdflib.term.Literal('925', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')), rdflib.term.Literal('619', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')), rdflib.term.Literal('2024-12-02T22:00:00', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#dateTime')))
(rdflib.term.Literal('FliS_T'), rdflib.term.Literal('727', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')), rdflib.term.Literal('1364', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')), rdflib.term.Literal('637', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')), rdflib.term.Literal('2024-12-02T19:00:00', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#dateTime')))
(rdflib.term.Literal('FliS_T'), rdflib.term.Literal('815', datatype=rdflib.term.URIRef('http:

In [11]:
import rdflib
from rdflib.namespace import RDF, RDFS, OWL, XSD, Namespace

# --- Define Namespaces ---
US = Namespace("https://smartcity.linkeddata.es/lcc/ontology/urban-sensors#")
SCHEMA = Namespace("http://schema.org/")
WGS84 = Namespace("http://www.w3.org/2003/01/geo/wgs84_pos#")
WD = Namespace("http://wikidata.org/entity/") # Prefijo para Wikidata

# --- Carga el grafo desde el archivo ---
g = rdflib.Graph()
try:
    # MODIFICADO: Cambiado format="nt" a format="turtle" para coincidir con la extensión .ttl
    g.parse("../rdf/pedestrian-counting-system-sensor-locations-enriched-updated-reconciliated.ttl", format="turtle")
    print(f"Grafo cargado con {len(g)} triples.\n")
except FileNotFoundError:
    print("Error: No se encontró el archivo de datos '../rdf/pedestrian-counting-system-sensor-locations-enriched-updated-reconciliated.ttl'. Asegúrate de que existe y está en el directorio correcto.")
except Exception as e:
    # Proporcionar más detalles sobre el error de parseo si ocurre
    print(f"Error al parsear el archivo Turtle: {e}")


# --- Consulta 2: Intermedia (Localizaciones en Melbourne usando owl:sameAs y Coordenadas) ---
# La consulta sigue siendo la versión de depuración sin rdfs:label
query2 = """
SELECT ?location ?lat ?long ?cityEntity
WHERE {
    ?location a us:Location ;
              us:inAdministrativeArea ?cityEntity ;
              wgs84_pos:lat ?lat ;
              wgs84_pos:long ?long .

    # Encuentra la entidad ciudad que es 'sameAs' Melbourne en Wikidata
    ?cityEntity owl:sameAs wd:Q3141 .
    # Se ha eliminado temporalmente -> rdfs:label ?cityName .
}
LIMIT 10
"""

print("--- Consulta 2 (Modificada): Localizaciones en Melbourne (vía Wikidata Q3141) y Coordenadas (Primeras 10) ---")
results2 = g.query(query2, initNs={"us": US, "wgs84_pos": WGS84, "owl": OWL, "wd": WD, "rdfs": RDFS})
results2_list = list(results2) # Convertir a lista para comprobar si está vacío
if not results2_list:
    print("No se encontraron resultados para localizaciones en Melbourne (Wikidata Q3141).")
else:
    for row in results2_list:
        print(f"Location: {row.location}, City Entity: {row.cityEntity}, Lat: {row.lat}, Long: {row.long}")
print("-" * 60)



Grafo cargado con 1177 triples.

--- Consulta 2 (Modificada): Localizaciones en Melbourne (vía Wikidata Q3141) y Coordenadas (Primeras 10) ---
Location: https://smartcity.linkeddata.es/lcc/resource/location/143, City Entity: https://smartcity.linkeddata.es/lcc/resource/city/Melbourne, Lat: -37.821728, Long: 144.95557015
Location: https://smartcity.linkeddata.es/lcc/resource/location/117, City Entity: https://smartcity.linkeddata.es/lcc/resource/city/Melbourne, Lat: -37.81629332, Long: 144.97090877
Location: https://smartcity.linkeddata.es/lcc/resource/location/107, City Entity: https://smartcity.linkeddata.es/lcc/resource/city/Melbourne, Lat: -37.81246271, Long: 144.95690188
Location: https://smartcity.linkeddata.es/lcc/resource/location/6, City Entity: https://smartcity.linkeddata.es/lcc/resource/city/Melbourne, Lat: -37.81911705, Long: 144.96558255
Location: https://smartcity.linkeddata.es/lcc/resource/location/49, City Entity: https://smartcity.linkeddata.es/lcc/resource/city/Melbou

In [12]:

# --- Consulta 3: Avanzada (Localizaciones en Calles con Wikidata, mostrando Wikidata de la Ciudad) ---
# Busca localizaciones en calles enlazadas a Wikidata y muestra también el enlace Wikidata de su ciudad
query3 = """
SELECT ?location ?lat ?long ?streetName ?wikidataStreet ?cityName ?wikidataCity
WHERE {
    # Localización y coordenadas
    ?location a us:Location ;
              wgs84_pos:lat ?lat ;
              wgs84_pos:long ?long ;
              # Enlace a la entidad calle
              us:hasAddress ?streetEntity ;
              # Enlace a la entidad ciudad
              us:inAdministrativeArea ?cityEntity .

    # Detalles de la calle y su enlace Wikidata
    ?streetEntity a schema:PostalAddress ;
                  schema:streetAddress ?streetName ;
                  owl:sameAs ?wikidataStreet .

    # Detalles de la ciudad y su enlace Wikidata
    ?cityEntity a schema:City ; # O schema:AdministrativeArea si es más general
                rdfs:label ?cityName ;
                owl:sameAs ?wikidataCity .

    # Filtros para asegurar que los enlaces son de Wikidata
    FILTER(STRSTARTS(STR(?wikidataStreet), "http://wikidata.org/entity/"))
    FILTER(STRSTARTS(STR(?wikidataCity), "http://wikidata.org/entity/"))
}
LIMIT 5 # Reducido el límite para que la salida sea más manejable
"""

print("--- Consulta 3: Localizaciones en Calles con Wikidata + Ciudad Wikidata (Primeras 5) ---")
results3 = g.query(query3, initNs={"us": US, "schema": SCHEMA, "wgs84_pos": WGS84, "owl": OWL, "rdfs": RDFS})
if not results3:
    print("No se encontraron resultados.")
else:
    for row in results3:
        print(f"Location: {row.location}\n  Coords: ({row.lat}, {row.long})")
        print(f"  Street: {row.streetName} ({row.wikidataStreet})")
        print(f"  City: {row.cityName} ({row.wikidataCity})\n")
print("-" * 60)


--- Consulta 3: Localizaciones en Calles con Wikidata + Ciudad Wikidata (Primeras 5) ---
Location: https://smartcity.linkeddata.es/lcc/resource/location/87
  Coords: (-37.80454949, 144.94921863)
  Street: Errol St/Victoria St (Melbourne City) (http://wikidata.org/entity/Q112824311)
  City: North Melbourne (http://wikidata.org/entity/Q7056074)

Location: https://smartcity.linkeddata.es/lcc/resource/location/117
  Coords: (-37.81629332, 144.97090877)
  Street: Flinders Street (http://wikidata.org/entity/Q3442521)
  City: Melbourne (http://wikidata.org/entity/Q3141)

Location: https://smartcity.linkeddata.es/lcc/resource/location/107
  Coords: (-37.81246271, 144.95690188)
  Street: William Street (http://wikidata.org/entity/Q2287373)
  City: Melbourne (http://wikidata.org/entity/Q3141)

Location: https://smartcity.linkeddata.es/lcc/resource/location/6
  Coords: (-37.81911705, 144.96558255)
  Street: Flinders Street (http://wikidata.org/entity/Q3442521)
  City: Melbourne (http://wikidata.o